# Step 9: Dose-Response Analysis for Quantity-Aware Health Risk Predictions

This notebook implements quantity-aware health risk predictions that consider:
- Daily usage amounts of ingredients/nutrients
- RDA (Recommended Daily Allowance) reference values
- Dose-response scaling for risk adjustment

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dose_response import (
    IngredientQuantity,
    calculate_dose_response,
    adjust_risk_predictions,
    parse_ingredient_string,
    RDA_DATABASE,
    NutrientType
)

print('Imports successful!')

## 2. View RDA Reference Values

In [ ]:
# Display all RDA values
rda_data = []
for key, rda in RDA_DATABASE.items():
    rda_data.append({
        'Nutrient': rda.name,
        'Daily Value': rda.value,
        'Unit': rda.unit,
        'Type': rda.nutrient_type.value,
        'Description': rda.description
    })

rda_df = pd.DataFrame(rda_data)
print(f'Loaded {len(rda_df)} RDA reference values from US FDA guidelines')
rda_df

## 3. Single Ingredient Analysis with Quantity

Let's analyze what happens when someone consumes 75g of sugar (150% of the daily limit of 50g).

In [ ]:
# Analyze sugar intake of 75g (150% of daily limit)
sugar = IngredientQuantity('sugar', 75, 'g')
result = calculate_dose_response(sugar)

print('=== DOSE-RESPONSE ANALYSIS ===')
print(f'Ingredient: {result.ingredient}')
print(f'Amount Consumed: {result.original_amount}{result.original_unit}')
print(f'Daily Limit (RDA): {result.rda_value}{result.rda_unit}')
print(f'Percentage of RDA: {result.pct_rda * 100:.1f}%')
print(f'Dose-Response Factor: {result.dose_response_factor:.3f}')
print(f'Risk Level: {result._get_risk_level().upper()}')
print()
print('Interpretation:')
print(f'  - Consuming {result.pct_rda*100:.0f}% of the daily limit')
print(f'  - Risk predictions will be scaled by {result.dose_response_factor:.2f}x')

## 4. Load GNN Base Predictions and Apply Dose Adjustment

In [ ]:
# Load GNN base predictions
gnn_df = pd.read_csv('../models/ingredient_risk_predictions.csv')
print(f'Loaded {len(gnn_df):,} ingredient predictions')
gnn_df.head()

In [ ]:
# Get base prediction for sugar
sugar_matches = gnn_df[gnn_df['ingredient'].str.lower().str.contains('sugar', na=False)]
if len(sugar_matches) > 0:
    sugar_row = sugar_matches.iloc[0]
    base_preds = {
        'diabetes': float(sugar_row['diabetes_risk']),
        'obesity': float(sugar_row['obesity_risk']),
        'hypertension': float(sugar_row['hypertension_risk']),
        'cancer': float(sugar_row['cancer_risk']),
        'cardiovascular': float(sugar_row['cardiovascular disease_risk'])
    }
else:
    # Default values if not found
    base_preds = {'diabetes': 0.5, 'obesity': 0.4, 'hypertension': 0.3, 'cancer': 0.2, 'cardiovascular': 0.35}

print('Base Predictions (without quantity consideration):')
for disease, prob in base_preds.items():
    print(f'  {disease.title()}: {prob:.1%}')

In [ ]:
# Adjust predictions based on 75g sugar (150% RDA)
adjusted = adjust_risk_predictions(base_preds, [result])

print('=== COMPARISON: Base vs Quantity-Adjusted Predictions ===')
print(f'Amount: 75g sugar (150% of daily limit)')
print(f'Dose-Response Factor: {result.dose_response_factor:.2f}x')
print()
print(f'{"Disease":<20} {"Base":>10} {"Adjusted":>10} {"Change":>10}')
print('-' * 50)
for disease in base_preds:
    base = base_preds[disease]
    adj = adjusted[disease]
    change_pct = ((adj - base) / base) * 100 if base > 0 else 0
    print(f'{disease.title():<20} {base:>9.1%} {adj:>9.1%} {change_pct:>+9.0f}%')

## 5. Multiple Ingredients with Quantities

The system can parse free-form text like "50g sugar, 3000mg sodium, 30g fiber"

In [ ]:
# Parse free-form text input
text = '50g sugar, 3000mg sodium, 30g fiber'
ingredients = parse_ingredient_string(text)

print(f'Input: "{text}"')
print('\n=== PARSED INGREDIENTS ==="')

dose_results = []
for ing in ingredients:
    dr = calculate_dose_response(ing)
    dose_results.append(dr)
    
    # Determine if exceeding limit or meeting target
    if dr.nutrient_type == NutrientType.LIMIT:
        status = 'EXCEEDS LIMIT' if dr.pct_rda > 1.0 else 'Within limit'
    elif dr.nutrient_type == NutrientType.TARGET:
        status = 'Meets target' if dr.pct_rda >= 1.0 else 'BELOW TARGET'
    else:
        status = 'Unknown'
    
    print(f'\n{ing.name.upper()}:')
    print(f'  Amount: {ing.amount}{ing.unit}')
    print(f'  RDA Reference: {dr.rda_value}{dr.rda_unit}')
    print(f'  Percentage of RDA: {dr.pct_rda*100:.0f}%')
    print(f'  Dose Factor: {dr.dose_response_factor:.2f}x')
    print(f'  Status: {status}')

## 6. Visualization: Risk vs Quantity (Dose-Response Curve)

In [ ]:
# Compare different sugar amounts
amounts = [10, 25, 50, 75, 100, 125]
diseases = list(base_preds.keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Dose-response factor curve
ax1 = axes[0]
pct_rda_values = np.linspace(0, 2.5, 100)  # 0% to 250% RDA
factors = [0.1 + 0.9 * (p ** 1.2) for p in pct_rda_values]
factors = [min(2.5, max(0.1, f)) for f in factors]

ax1.plot(pct_rda_values * 100, factors, 'b-', linewidth=2)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Baseline (1.0x)')
ax1.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='100% RDA')
ax1.set_xlabel('Percentage of Daily Limit (%)')
ax1.set_ylabel('Dose-Response Factor')
ax1.set_title('Dose-Response Scaling Function (LIMIT nutrients)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 250)
ax1.set_ylim(0, 2.6)

# Right plot: Adjusted risks by quantity
ax2 = axes[1]
x = np.arange(len(diseases))
width = 0.12
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(amounts)))

for i, amt in enumerate(amounts):
    ing = IngredientQuantity('sugar', amt, 'g')
    dr = calculate_dose_response(ing)
    adj = adjust_risk_predictions(base_preds, [dr])
    values = [adj[d] for d in diseases]
    ax2.bar(x + i*width, values, width, label=f'{amt}g ({dr.pct_rda*100:.0f}%)', color=colors[i])

ax2.set_ylabel('Risk Probability')
ax2.set_title('Health Risk by Sugar Quantity')
ax2.set_xticks(x + width * 2.5)
ax2.set_xticklabels([d.title() for d in diseases], rotation=45, ha='right')
ax2.legend(title='Sugar Amount', bbox_to_anchor=(1.02, 1), loc='upper left')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../models/dose_response_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved visualization to ../models/dose_response_visualization.png')

## 7. Interactive Demo: Analyze Your Own Ingredients

In [ ]:
def analyze_intake(ingredient_text):
    '''
    Analyze ingredient intake and show dose-response adjusted risks.
    
    Usage: analyze_intake('50g sugar, 2000mg sodium')
    '''
    ingredients = parse_ingredient_string(ingredient_text)
    
    print(f'\n{"="*60}')
    print(f'ANALYZING: {ingredient_text}')
    print('='*60)
    
    all_dose_results = []
    combined_base = {}
    
    for ing in ingredients:
        # Get dose response
        dr = calculate_dose_response(ing)
        all_dose_results.append(dr)
        
        # Try to get GNN predictions
        matches = gnn_df[gnn_df['ingredient'].str.lower().str.contains(ing.name.lower(), na=False)]
        if len(matches) > 0:
            row = matches.iloc[0]
            preds = {
                'diabetes': float(row['diabetes_risk']),
                'obesity': float(row['obesity_risk']),
                'hypertension': float(row['hypertension_risk']),
                'cancer': float(row['cancer_risk']),
                'cardiovascular': float(row['cardiovascular disease_risk'])
            }
            for d, p in preds.items():
                combined_base[d] = max(combined_base.get(d, 0), p)
        
        # Print ingredient analysis
        status_emoji = '⚠️' if dr.pct_rda > 1.0 else '✅' if dr.pct_rda >= 0.5 else '📉'
        print(f'\n{status_emoji} {ing.name.upper()}: {ing.amount}{ing.unit}')
        if dr.is_known_nutrient:
            print(f'   {dr.pct_rda*100:.0f}% of daily {"limit" if dr.nutrient_type==NutrientType.LIMIT else "target"}')
            print(f'   Risk scaling: {dr.dose_response_factor:.2f}x')
    
    # Show adjusted predictions
    if combined_base:
        adjusted = adjust_risk_predictions(combined_base, all_dose_results)
        print(f'\n{"─"*60}')
        print('RISK PREDICTIONS (Quantity-Adjusted):')
        print('─'*60)
        for disease in sorted(adjusted.keys(), key=lambda d: adjusted[d], reverse=True):
            level = 'HIGH' if adjusted[disease] > 0.6 else 'MODERATE' if adjusted[disease] > 0.3 else 'LOW'
            bar = '█' * int(adjusted[disease] * 20)
            print(f'{disease.title():20} {adjusted[disease]:6.1%} [{bar:<20}] {level}')
    
    return all_dose_results

In [ ]:
# Example: Analyze a high-risk intake
analyze_intake('100g sugar, 4000mg sodium, 5g fiber')

In [ ]:
# Example: Analyze a healthier intake
analyze_intake('25g sugar, 1500mg sodium, 30g fiber, 60g protein')

## Summary

The dose-response system now:

1. **Accepts quantities** - Users can specify amounts like "50g sugar" or "200mg sodium"

2. **Calculates % of RDA** - Compares intake to US FDA recommended daily values

3. **Applies dose-response scaling** - Risk increases non-linearly with overconsumption

4. **Adjusts GNN predictions** - Base predictions are scaled by the dose-response factor

### Key Files Created:
- `src/dose_response.py` - Core dose-response calculation module
- `src/api.py` - Updated with `/api/analyze/v2` endpoint

### API Usage:
```python
# POST /api/analyze/v2
{
    "ingredients": [
        {"name": "sugar", "amount": 50, "unit": "g"},
        {"name": "sodium", "amount": 2000, "unit": "mg"}
    ]
}
```